# 💻 **Práctica Profesionalizante 2: Proyecto Final**

## 📚 **Datos**

👨‍💻 Integrantes:

- Garcia Alves, Andrés (DNI 30182100)
- Vega, Horacio Facundo (DNI 36676147)

👨‍🏫 Profesor:

- Paredes Rojas, Julio

🚑 Temática: **Accidentes Viales**

✍🏻 Título: Detección de Patrones en Accidentes en la Vía Pública

📦 Conjunto de datos:
  - Datos oficiales de CABA en Seguridad Vial
	  - Información sobre siniestros viales ocurridos en la Ciudad desde el año 2019 hasta el 2024.
  - URLs:
    - https://data.buenosaires.gob.ar/dataset/victimas-siniestros-viales
    - https://data.buenosaires.gob.ar/dataset/victimas-siniestros-viales/resource/d66f63c6-e9b2-47b2-80b2-58a0cce0ca86
    - https://data.buenosaires.gob.ar/dataset/victimas-siniestros-viales/resource/79914119-0e1e-47c6-9f7b-f1f7bf542786


## 📝 **Consignas**

* Primera Parte (Proyecto de Ciencia de Datos)

	- Objetivos
	- Introducción
	- Marco Conceptual
	- Bloques de código bien comentados
	- Resultados
	- Conclusiones

* Segunda Parte (Para Publicación en una Revista Científica)

	- Portada
	- Autor
	- Introducción
	- Objetivos
	- Hipótesis
	- Marco conceptual, estilo revista técnica (mirar como esta acá tipo estilo APA)
	- Metodología del proyecto
	- 6 referencias en estilo APA promedio, con link verificable dentro del HTML

* Tecnologías a Incorporar:

	- HTML Interactivo (Gradio)
	- Ngrok
	- MLFlow

## 💡 **Abstract**

En este proyecto se analiza el conjunto de datos de **Siniestros Viales** de la Ciudad Autónoma de Buenos Aires (CABA), proporcionado por el Observatorio de Movilidad y Seguridad Vial del GCBA, con cobertura del período **2019–2024**.

El objetivo principal es **detectar patrones** en la ocurrencia y gravedad de los accidentes de tránsito mediante técnicas de Ciencia de Datos.

Respondiendo preguntas clave como:
- ¿en qué horarios y comunas se concentran los siniestros más graves?
- ¿qué combinaciones de vehículos generan mayor riesgo?
- ¿cómo impactó la pandemia de COVID-19 en la siniestralidad?

El conjunto de datos está compuesto por dos archivos en formato CSV:
- **`siniestros_viales_hechos.csv`**:
	- un registro por siniestro (perfil de participantes, gravedad, tipo de vía)
	- 54.064 filas × 21 columnas
  
- **`siniestros_viales_victimas.csv`**:
	- un registro por víctima (perfil demográfico, rol en el siniestro, gravedad individual, fecha de fallecimiento)
  - 62.076 filas × 9 columnas

El pipeline completo incluye:
- limpieza de datos
- ingeniería de variables
- análisis exploratorio (EDA) con visualizaciones interactivas (Plotly)
- un modelo de clasificación con tracking vía MLFlow
- y una demo interactiva con Gradio y Ngrok.

## ❓ **Preguntas de Investigación e Hipótesis**

1. **Franjas horarias y días de mayor riesgo**
   - ¿En qué franjas horarias y días de la semana se concentran los siniestros de mayor gravedad?
   - *Hipótesis:* La madrugada de viernes a domingo concentra mayor proporción de siniestros graves/mortales.

2. **Distribución geográfica por comunas**
   - ¿Qué comunas acumulan mayor cantidad de siniestros? ¿Difiere la gravedad entre comunas?
   - *Hipótesis:* Las comunas con mayor flujo vehicular (1, 4, 9) [1] muestran más siniestros totales, con presencia relevante de siniestros mortales.

3. **Combinaciones de participantes más peligrosas**
   - ¿Cuál es la combinación víctima-contraparte que produce más siniestros graves/mortales?
   - *Hipótesis:* Las combinaciones MOTO-AUTO y PEATON-AUTO concentran la mayor cantidad de víctimas graves y mortales.

4. **Evolución temporal 2019–2024**
   - ¿Cómo varió la cantidad de siniestros a lo largo del tiempo? ¿Se observa el efecto de la pandemia COVID-19?
   - *Hipótesis:* El año 2020 muestra una caída pronunciada en siniestros leves (menor circulación), con recuperación progresiva hasta 2024.

5. **Perfil demográfico de las víctimas**
   - ¿Qué grupos etarios y qué género presentan mayor exposición al riesgo?
   - *Hipótesis:* Los hombres jóvenes (18-30 años) concentran la mayor proporción de víctimas mortales.

6. **Tipo de vía y gravedad**
   - ¿Los siniestros en autopistas y avenidas son más graves que en calles secundarias?
   - *Hipótesis:* Los siniestros en autopistas presentan mayor proporción de víctimas mortales por efecto de la velocidad.

Nota [1]:
- Comuna 1: Retiro, San Nicolás, Puerto Madero, San Telmo, Montserrat y Constitución.
- Comuna 4: La Boca, Barracas, Parque Patricios y Nueva Pompeya.
- Comuna 9: Liniers, Mataderos y Parque Avellaneda.

## ⚙️ **Preprocesamiento**

Se procede en esta sección a:
- Importar de las librerías a utilizar
- Detectar el entorno de ejecución (VSCode local vs Google Colab) y configurar de las rutas a los datasets
- Cargar de los datasets
- Inspección inicial de sus estructuras
- Validación por nulos (+ imputación en caso de haberlos)
- Verificar de duplicados
- Conversión de tipos de datos
- Ingeniería de variables (feature engineering)
- Merge de ambos datasets

#### Importación de librerías ...

In [1]:
# librerías generales
import os
import re
import sys
import warnings

In [2]:
# librerías de análisis y visualización
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
# configuraciones generales
warnings.filterwarnings('ignore')
np.set_printoptions(linewidth=500)
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)
pd.set_option("display.width", None)
pd.set_option("display.float_format", "{:.2f}".format)

#### Detección de entorno y configuración de rutas ...

In [4]:
# Detectar si se ejecuta en Google Colab o localmente
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/gdrive', force_remount=False)

    # ajustar según la ubicación de los datasets en el Google Drive
    BASE_PATH = '/content/gdrive/MyDrive/Colab Notebooks/ISSD-PP2/Proyecto Final/Datasets'
    IS_GOOGLE_COLAB = True

except ImportError:

    # path relativo desde el archivo actual + '/Datasets'
    BASE_PATH = os.path.join(os.getcwd(), 'Datasets')
    IS_GOOGLE_COLAB = False

print(f"Entorno  : {'Google Colab' if IS_GOOGLE_COLAB else 'Local (VS Code)'}")
print(f"Ruta base: {BASE_PATH}")

Entorno  : Local (VS Code)
Ruta base: c:\Users\Andres\Desktop\ISSD\PP2 - Pract. Profesionalizante 2\Source Code\Datasets


#### Carga de los datasets ...

In [5]:
path_hechos = os.path.join(BASE_PATH, 'dataset siniestros_viales_hechos.csv')
path_victimas = os.path.join(BASE_PATH, 'dataset siniestros_viales_victimas.csv')

df_hechos = pd.read_csv(path_hechos, sep=';', low_memory=False)
df_victimas = pd.read_csv(path_victimas, sep=';', low_memory=False)

print(f"df_hechos  : {df_hechos.shape[0]} filas × {df_hechos.shape[1]} columnas")
print(f"df_victimas: {df_victimas.shape[0]} filas × {df_victimas.shape[1]} columnas")

df_hechos  : 54064 filas × 21 columnas
df_victimas: 62076 filas × 9 columnas


#### Inspección inicial ...

In [6]:
print("- Hechos (top 3)")
df_hechos.head(3)

- Hechos (top 3)


,id_siniestro,numero_total_de_victimas,numero_victimas_leve_siniestro,numero_victimas_grave_siniestro,numero_victimas_mortal_siniestro,fecha_siniestro,anio_siniestro,mes_siniestro,dia_siniestro,hora_siniestro,rango_horario,direccion_normalizada_siniestro,comuna_siniestro,tipo_de_via_siniestro,geocodificacion_plana,longitud_siniestro,latitud_siniestro,participantes_siniestro,modo_desplazamiento_victima,contraparte_siniestro,gravedad_siniestro
0,LC-2019-0000445,3,3,0,0,2019-01-01,2019,1,1,13:40:00,13.00,NaN,Comuna 11,NaN,POINT (94371.44237885093025398 102884.19365105...,-58.52,-34.60,AUTO-AUTO,AUTO,AUTO,LEVE
1,LC-2019-0000194,1,1,0,0,2019-01-01,2019,1,1,07:15:00,7.00,NaN,Comuna 9,NaN,POINT (94030.76669932194636203 97681.071761248...,-58.53,-34.65,AUTO-CAMION,AUTO,CAMION,LEVE
2,LC-2019-0000329,1,1,0,0,2019-01-01,2019,1,1,12:24:00,12.00,NaN,Comuna 4,NaN,POINT (105367.86665769023238681 97877.75085908...,-58.40,-34.65,AUTO-MOVIL,AUTO,MOVIL,LEVE


In [7]:
print("- Hechos (tipos de datos)")
df_hechos.dtypes	# tipos de datos

- Hechos (tipos de datos)


id_siniestro                         object
numero_total_de_victimas              int64
numero_victimas_leve_siniestro        int64
numero_victimas_grave_siniestro       int64
numero_victimas_mortal_siniestro      int64
fecha_siniestro                      object
anio_siniestro                        int64
mes_siniestro                         int64
dia_siniestro                         int64
hora_siniestro                       object
rango_horario                       float64
direccion_normalizada_siniestro      object
comuna_siniestro                     object
tipo_de_via_siniestro                object
geocodificacion_plana                object
longitud_siniestro                  float64
latitud_siniestro                   float64
participantes_siniestro              object
modo_desplazamiento_victima          object
contraparte_siniestro                object
gravedad_siniestro                   object
dtype: object

In [8]:
print("- Hechos (estadísticas descriptivas)")
df_hechos.describe(include='all')

- Hechos (estadísticas descriptivas)


,id_siniestro,numero_total_de_victimas,numero_victimas_leve_siniestro,numero_victimas_grave_siniestro,numero_victimas_mortal_siniestro,fecha_siniestro,anio_siniestro,mes_siniestro,dia_siniestro,hora_siniestro,rango_horario,direccion_normalizada_siniestro,comuna_siniestro,tipo_de_via_siniestro,geocodificacion_plana,longitud_siniestro,latitud_siniestro,participantes_siniestro,modo_desplazamiento_victima,contraparte_siniestro,gravedad_siniestro
count,54064,54064.00,54064.00,54064.00,54064.00,54064,54064.00,54064.00,54064.00,53987,53987.00,41164,51047,41837,51697,51686.00,51686.00,54064,54064,54064,54064
unique,54064,NaN,NaN,NaN,NaN,2192,NaN,NaN,NaN,1464,NaN,21688,18,4,32216,NaN,NaN,189,13,22,3
top,LC-2019-0000445,NaN,NaN,NaN,NaN,2024-12-17,NaN,NaN,NaN,18:00:00,NaN,"PAZ, GRAL. AV. y BALBIN, RICARDO, DR. AV.",Comuna 1,AVENIDA,POINT(99998.01828 107775.7957),NaN,NaN,MOTO-AUTO,MOTO,AUTO,LEVE
freq,1,NaN,NaN,NaN,NaN,63,NaN,NaN,NaN,880,NaN,109,5790,24480,423,NaN,NaN,11921,19809,23657,51199
mean,NaN,1.15,1.09,0.05,0.01,NaN,2021.64,6.71,15.56,NaN,13.63,NaN,NaN,NaN,NaN,-58.44,-34.61,NaN,NaN,NaN,NaN
std,NaN,0.56,0.59,0.23,0.11,NaN,1.77,3.42,8.74,NaN,5.54,NaN,NaN,NaN,NaN,0.04,0.03,NaN,NaN,NaN,NaN
min,NaN,1.00,0.00,0.00,0.00,NaN,2019.00,1.00,1.00,NaN,0.00,NaN,NaN,NaN,NaN,-58.53,-34.71,NaN,NaN,NaN,NaN
25%,NaN,1.00,1.00,0.00,0.00,NaN,2020.00,4.00,8.00,NaN,10.00,NaN,NaN,NaN,NaN,-58.48,-34.63,NaN,NaN,NaN,NaN
50%,NaN,1.00,1.00,0.00,0.00,NaN,2022.00,7.00,15.00,NaN,14.00,NaN,NaN,NaN,NaN,-58.44,-34.61,NaN,NaN,NaN,NaN
75%,NaN,1.00,1.00,0.00,0.00,NaN,2023.00,10.00,23.00,NaN,18.00,NaN,NaN,NaN,NaN,-58.41,-34.59,NaN,NaN,NaN,NaN


In [9]:
print("Victimas (top 3)")
df_victimas.head(3)

Victimas (top 3)


,id_siniestro,fecha_siniestro,anio_siniestro,modo_desplazamiento_victima,sexo_victima,edad_victima,GRAVEdad_victima,rol_victima,fecha_fallecimiento_victima
0,LC-2019-0000647,2019-01-01,2019,MOTO,M,54,GRAVE,SD,NaN
1,LC-2019-0000600,2019-01-01,2019,SD,F,1,LEVE,SD,NaN
2,LC-2019-0000136,2019-01-01,2019,SD,F,21,LEVE,SD,NaN


In [10]:
print("- Victimas (tipos de datos)")
df_victimas.dtypes

- Victimas (tipos de datos)


id_siniestro                   object
fecha_siniestro                object
anio_siniestro                  int64
modo_desplazamiento_victima    object
sexo_victima                   object
edad_victima                   object
GRAVEdad_victima               object
rol_victima                    object
fecha_fallecimiento_victima    object
dtype: object

In [11]:
print("- Victimas (estadísticas descriptivas)")
df_victimas.describe(include='all')

- Victimas (estadísticas descriptivas)


,id_siniestro,fecha_siniestro,anio_siniestro,modo_desplazamiento_victima,sexo_victima,edad_victima,GRAVEdad_victima,rol_victima,fecha_fallecimiento_victima
count,62076,62076,62076.00,62076,62076,62076,62076,61865,610
unique,54064,2192,NaN,18,3,102,3,8,531
top,LC-2024-0243962,2024-12-17,NaN,SD,M,SD,LEVE,SD,2024-4-4
freq,26,86,NaN,21874,33666,16755,58995,49983,4
mean,NaN,NaN,2021.64,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,1.78,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,2019.00,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,2020.00,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,2022.00,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,2023.00,NaN,NaN,NaN,NaN,NaN,NaN


#### Validación e imputación de nulos ...

In [12]:
def tabla_nulos(df):
	nulos = df.isnull().sum()									# conteo de nulos
	pct   = (nulos / len(df) * 100).round(2)	# porcentaje de nulos
	return pd.DataFrame({'nulos': nulos, '%': pct}).query('nulos > 0').sort_values('nulos', ascending=False)

In [13]:
print("- Nulos en df_hechos")
print(tabla_nulos(df_hechos))


- Nulos en df_hechos
                                 nulos     %
direccion_normalizada_siniestro  12900 23.86
tipo_de_via_siniestro            12227 22.62
comuna_siniestro                  3017  5.58
longitud_siniestro                2378  4.40
latitud_siniestro                 2378  4.40
geocodificacion_plana             2367  4.38
hora_siniestro                      77  0.14
rango_horario                       77  0.14


In [14]:
# Hechos

# Tipo de vía: completar sin datos con 'SD' (convención del dataset)
df_hechos['tipo_de_via_siniestro'] = df_hechos['tipo_de_via_siniestro'].fillna('SD')

# Coordenadas: forzar a numérico (algunos registros tienen cadena vacía)
df_hechos['longitud_siniestro'] = pd.to_numeric(df_hechos['longitud_siniestro'], errors='coerce')
df_hechos['latitud_siniestro']  = pd.to_numeric(df_hechos['latitud_siniestro'],  errors='coerce')

# Franja horaria: forzar a numérico
df_hechos['rango_horario'] = pd.to_numeric(df_hechos['rango_horario'], errors='coerce')

print("Nulos restantes:", df_hechos[['tipo_de_via_siniestro', 'longitud_siniestro', 'latitud_siniestro', 'rango_horario']].isnull().sum().to_dict())

Nulos restantes: {'tipo_de_via_siniestro': 0, 'longitud_siniestro': 2378, 'latitud_siniestro': 2378, 'rango_horario': 77}


In [15]:
print("- Nulos en df_victimas")
print(tabla_nulos(df_victimas))

- Nulos en df_victimas


                             nulos     %
fecha_fallecimiento_victima  61466 99.02
rol_victima                    211  0.34


In [16]:
# Victimas

# Edad: forzar a numérico (hay valores 'SD' o vacíos)
df_victimas['edad_victima'] = pd.to_numeric(df_victimas['edad_victima'], errors='coerce')

# Sexo: rellenar vacíos con 'SD'
df_victimas['sexo_victima'] = df_victimas['sexo_victima'].fillna('SD')

print("Nulos restantes df_victimas:", df_victimas[['edad_victima', 'sexo_victima']].isnull().sum().to_dict())

Nulos restantes df_victimas: {'edad_victima': 16756, 'sexo_victima': 0}


#### Verificación de duplicados ...

In [17]:
dup_hechos   = df_hechos.duplicated(subset='id_siniestro').sum()
dup_victimas = df_victimas.duplicated().sum()

print(f"Duplicados en df_hechos   (por id_siniestro): {dup_hechos}")
print(f"Duplicados en df_victimas (fila completa):    {dup_victimas}")

Duplicados en df_hechos   (por id_siniestro): 0
Duplicados en df_victimas (fila completa):    1159


#### Conversión de tipos de datos ...

In [18]:
# renombrar columnas de victimas (por simplicidad)
rename_map = {
    'id_siniestro': 'id_hecho',
    'fecha_siniestro': 'fecha',
    'GRAVEdad_victima': 'gravedad',
    'fecha_fallecimiento_victima': 'fecha_fallecimiento',
    'modo_desplazamiento_victima': 'victima'
}

for old_col, new_col in rename_map.items():
    if old_col in df_victimas.columns and new_col not in df_victimas.columns:
        df_victimas.rename(columns={old_col: new_col}, inplace=True)

In [19]:
# convertir fechas a datetime
df_hechos['fecha_siniestro'] = pd.to_datetime(df_hechos['fecha_siniestro'], errors='coerce')

if 'fecha' in df_victimas.columns:
    df_victimas['fecha'] = pd.to_datetime(df_victimas['fecha'], errors='coerce')

if 'fecha_fallecimiento' in df_victimas.columns:
    df_victimas['fecha_fallecimiento'] = pd.to_datetime(df_victimas['fecha_fallecimiento'], errors='coerce')

In [20]:
# extraer dia de la semana
DIAS = {0: 'Lunes', 1: 'Martes', 2: 'Miercoles', 3: 'Jueves', 4: 'Viernes', 5: 'Sabado', 6: 'Domingo'}
df_hechos['dia_semana'] = df_hechos['fecha_siniestro'].dt.dayofweek
df_hechos['dia_nombre'] = df_hechos['dia_semana'].map(DIAS)

In [21]:
# estandarizar gravedad y sexo a mayusculas
if 'gravedad_siniestro' in df_hechos.columns:
    df_hechos['gravedad_siniestro'] = df_hechos['gravedad_siniestro'].astype(str).str.upper().fillna('SD')

if 'gravedad' in df_victimas.columns:
    df_victimas['gravedad'] = df_victimas['gravedad'].astype(str).str.upper().fillna('SD')

if 'sexo_victima' in df_victimas.columns:
    df_victimas['sexo_victima'] = df_victimas['sexo_victima'].astype(str).str.upper().fillna('SD')

In [22]:
# resultados
print('Columnas normalizadas de df_victimas:')
print(df_victimas.columns.tolist())
print('\n')

print('Tipos actualizados:')		# con un top-3 de ejemplo
print(df_hechos[['fecha_siniestro', 'dia_semana', 'dia_nombre', 'gravedad_siniestro']].head(3).to_string())

Columnas normalizadas de df_victimas:
['id_hecho', 'fecha', 'anio_siniestro', 'victima', 'sexo_victima', 'edad_victima', 'gravedad', 'rol_victima', 'fecha_fallecimiento']


Tipos actualizados:
  fecha_siniestro  dia_semana dia_nombre gravedad_siniestro
0      2019-01-01           1     Martes               LEVE
1      2019-01-01           1     Martes               LEVE
2      2019-01-01           1     Martes               LEVE


#### Feature engineering (variables construidas con criterio) ...

In [23]:
# Hechos

# es_fin_de_semana (bool): True si el siniestro ocurrió sábado (5) o domingo (6)
df_hechos['es_fin_de_semana'] = df_hechos['dia_semana'].isin([5, 6])

# franja_del_dia: categoría según la hora del siniestro
def asignar_franja(hora):
    if pd.isna(hora):  return 'SD'
    h = int(hora)
    if   0 <= h <= 5:  return 'Madrugada (0-5h)'
    elif 6 <= h <= 11: return 'Mañana (6-11h)'
    elif 12 <= h <= 17: return 'Tarde (12-17h)'
    else:               return 'Noche (18-23h)'

df_hechos['franja_del_dia'] = df_hechos['rango_horario'].apply(asignar_franja)

# es_mortal (bool): True si alguna víctima del hecho falleció
df_hechos['es_mortal'] = df_hechos['gravedad_siniestro'] == 'MORTAL'

# n_comuna (int): número entero de la comuna (ejemplo "Comuna 11" pasa a ser 11)
def extraer_n_comuna(s):
    if pd.isna(s): return np.nan
    m = re.search(r'\d+', str(s))
    return int(m.group()) if m else np.nan

df_hechos['n_comuna'] = df_hechos['comuna_siniestro'].apply(extraer_n_comuna)

In [24]:
print("=== Nuevas variables en df_hechos ===")		# con un top-3 de ejemplo 
print(df_hechos[['id_siniestro','es_fin_de_semana','franja_del_dia','es_mortal','n_comuna']].head(3).to_string())

=== Nuevas variables en df_hechos ===
      id_siniestro  es_fin_de_semana  franja_del_dia  es_mortal  n_comuna
0  LC-2019-0000445             False  Tarde (12-17h)      False     11.00
1  LC-2019-0000194             False  Mañana (6-11h)      False      9.00
2  LC-2019-0000329             False  Tarde (12-17h)      False      4.00


In [25]:
# Victimas

# edad_grupo: bins por rango etario
bins   = [0, 17, 30, 60, 150]
labels = ['Menor (<18)', 'Joven (18-30)', 'Adulto (30-60)', 'Mayor (>60)']
df_victimas['edad_grupo'] = pd.cut(df_victimas['edad_victima'], bins=bins, labels=labels, right=True)

# tiempo_hasta_fallecimiento: días entre el siniestro y el fallecimiento (solo víctimas mortales)
df_victimas['tiempo_hasta_fallecimiento'] = (
    df_victimas['fecha_fallecimiento'] - df_victimas['fecha']
).dt.days

In [26]:
print("\n=== Nuevas variables en df_victimas ===")		# con un top-3 de ejemplo 
print(df_victimas[['id_hecho', 'edad_victima', 'edad_grupo', 'tiempo_hasta_fallecimiento']].head(3).to_string())


=== Nuevas variables en df_victimas ===
          id_hecho  edad_victima      edad_grupo  tiempo_hasta_fallecimiento
0  LC-2019-0000647         54.00  Adulto (30-60)                         NaN
1  LC-2019-0000600          1.00     Menor (<18)                         NaN
2  LC-2019-0000136         21.00   Joven (18-30)                         NaN


#### Merge de Hechos + Víctimas ...

In [27]:
# Left join: preservar todos los Hechos, sumándole los datos de Víctimas
# La clave de unión es id_siniestro (hechos) = id_hecho (victimas)
df = df_hechos.merge(
	df_victimas,
	left_on='id_siniestro', right_on='id_hecho',
	how='left',
	suffixes=('_hecho', '_victima')
)

In [28]:
print(f"df (con merge): {df.shape[0]} filas × {df.shape[1]} columnas")
print("\n")

print("Muestra:")
print(df[['id_siniestro','fecha_siniestro','gravedad_siniestro','sexo_victima','edad_victima','edad_grupo']].head(5).to_string())

df (con merge): 62076 filas × 38 columnas


Muestra:
      id_siniestro fecha_siniestro gravedad_siniestro sexo_victima  edad_victima      edad_grupo
0  LC-2019-0000445      2019-01-01               LEVE            F         42.00  Adulto (30-60)
1  LC-2019-0000445      2019-01-01               LEVE            M         37.00  Adulto (30-60)
2  LC-2019-0000445      2019-01-01               LEVE            M         42.00  Adulto (30-60)
3  LC-2019-0000194      2019-01-01               LEVE            F         33.00  Adulto (30-60)
4  LC-2019-0000329      2019-01-01               LEVE            M         23.00   Joven (18-30)


## 📊 **Análisis Exploratorio de Datos (EDA)**

Se analiza visual y estadísticamente cada una de las preguntas de investigación planteadas.

### 1️⃣ Pregunta 01 - Franjas horarias y días de mayor riesgo

- ¿En qué franjas horarias y días de la semana se concentran los siniestros de mayor gravedad?
- *Hipótesis:* La madrugada de viernes a domingo concentra mayor proporción de siniestros graves/mortales.

##### 1.A - Heatmap: siniestros por hora x día de la semana

In [29]:
# Heatmap de densidad: Hora del día vs Día de la semana
DIAS_ORDEN = ['Lunes','Martes','Miércoles','Jueves','Viernes','Sábado','Domingo']

pivot = (df_hechos
	.dropna(subset=['rango_horario','dia_nombre'])
	.groupby(['rango_horario','dia_nombre'])
	.size()
	.reset_index(name='cantidad')
)

pivot['rango_horario'] = pivot['rango_horario'].astype(int)

fig = px.density_heatmap(
	pivot, x='rango_horario', y='dia_nombre',
	z='cantidad',
	color_continuous_scale='YlOrRd',
	category_orders={'dia_nombre': DIAS_ORDEN},
	labels={'rango_horario': 'Hora del día', 'dia_nombre': 'Día', 'cantidad': 'Siniestros'},
	title='Distribución de Siniestros por Hora y Día de la Semana (2019-2024)'
)
fig.update_layout(height=420, xaxis=dict(tickmode='linear', tick0=0, dtick=1))
fig.show()

##### 1.B - Siniestros por franja horaria según gravedad

In [30]:
FRANJA_ORDEN = ['Madrugada (0-5h)', 'Mañana (6-11h)', 'Tarde (12-17h)', 'Noche (18-23h)']

franja_grav = (df_hechos[df_hechos['franja_del_dia'] != 'SD']
    .groupby(['franja_del_dia', 'gravedad_siniestro'])
    .size()
    .reset_index(name='cantidad')
)

fig = px.bar(
    franja_grav, x='franja_del_dia', y='cantidad', color='gravedad_siniestro',
    barmode='group',
    category_orders={
        'franja_del_dia': FRANJA_ORDEN,
        'gravedad_siniestro': ['LEVE', 'GRAVE', 'MORTAL', 'SD']
    },
    color_discrete_map={'LEVE': '#2196F3', 'GRAVE': '#FF9800', 'MORTAL': '#F44336', 'SD': '#9E9E9E'},
    labels={'franja_del_dia': 'Franja Horaria', 'cantidad': 'Siniestros', 'gravedad_siniestro': 'Gravedad'},
    title='Siniestros por Franja Horaria según Gravedad'
)
fig.update_layout(height=420)
fig.show()

### 2️⃣ Pregunta 02 - Distribución geográfica por comunas

- ¿Qué comunas acumulan mayor cantidad de siniestros? ¿Difiere la gravedad entre comunas?
- *Hipótesis:* Las comunas con mayor flujo vehicular (1, 4, 9) muestran más siniestros totales.

##### 2.A - Siniestros por comuna según gravedad

In [31]:
comuna_grav = (df_hechos[df_hechos['n_comuna'].notna()]
    .assign(n_comuna=lambda d: d['n_comuna'].astype(int))
    .groupby(['n_comuna', 'gravedad_siniestro'])
    .size()
    .reset_index(name='cantidad')
    .sort_values('n_comuna')
)
comuna_grav['comuna_label'] = 'C' + comuna_grav['n_comuna'].astype(str).str.zfill(2)

fig = px.bar(
    comuna_grav, x='comuna_label', y='cantidad', color='gravedad_siniestro',
    barmode='stack',
    color_discrete_map={'LEVE': '#2196F3', 'GRAVE': '#FF9800', 'MORTAL': '#F44336', 'SD': '#9E9E9E'},
    category_orders={'gravedad_siniestro': ['LEVE', 'GRAVE', 'MORTAL', 'SD']},
    labels={'comuna_label': 'Comuna', 'cantidad': 'Siniestros', 'gravedad_siniestro': 'Gravedad'},
    title='Siniestros por Comuna y Gravedad - CABA 2019-2024'
)
fig.update_layout(height=440)
fig.show()

##### 2.B - Mapa de puntos (geolocalización de siniestros)

In [32]:
df_geo = df_hechos.dropna(subset=['latitud_siniestro', 'longitud_siniestro']).copy()
df_geo = df_geo[df_geo['gravedad_siniestro'].isin(['LEVE', 'GRAVE', 'MORTAL'])]

# muestra de hasta 10.000 puntos para no saturar el mapa
muestra = df_geo.sample(min(10_000, len(df_geo)), random_state=42)

fig = px.scatter_mapbox(
    muestra,
    lat='latitud_siniestro', lon='longitud_siniestro',
    color='gravedad_siniestro',
    color_discrete_map={'LEVE': '#2196F3', 'GRAVE': '#FF9800', 'MORTAL': '#F44336'},
    opacity=0.5, zoom=11, height=520,
    mapbox_style='open-street-map',
    title='Mapa de Siniestros en CABA (muestra de 10.000 hechos)',
    labels={'gravedad_siniestro': 'Gravedad'},
    hover_data=['fecha_siniestro', 'participantes_siniestro', 'comuna_siniestro']
)
fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))
fig.show()

### 3️⃣ Pregunta 03 - Combinaciones de participantes más peligrosas

- ¿Cuál es la combinación víctima-contraparte que produce más siniestros graves/mortales?
- *Hipótesis:* MOTO-AUTO y PEATON-AUTO concentran la mayor cantidad de víctimas graves y mortales.

##### 3.A - Top 15 combinaciones de participantes por gravedad

In [33]:
top15_part = (df_hechos
    .groupby('participantes_siniestro')
    .size()
    .nlargest(15)
    .index
    .tolist()
)

part_counts = (df_hechos[df_hechos['participantes_siniestro'].isin(top15_part)]
    .groupby(['participantes_siniestro', 'gravedad_siniestro'])
    .size()
    .reset_index(name='cantidad')
)

fig = px.bar(
    part_counts, x='participantes_siniestro', y='cantidad', color='gravedad_siniestro',
    barmode='stack',
    color_discrete_map={'LEVE': '#2196F3', 'GRAVE': '#FF9800', 'MORTAL': '#F44336', 'SD': '#9E9E9E'},
    category_orders={'gravedad_siniestro': ['LEVE', 'GRAVE', 'MORTAL', 'SD']},
    labels={'participantes_siniestro': 'Participantes', 'cantidad': 'Siniestros', 'gravedad_siniestro': 'Gravedad'},
    title='Top 15 Combinaciones de Participantes por Gravedad'
)
fig.update_xaxes(tickangle=40)
fig.update_layout(height=480)
fig.show()

##### 3.B - Proporción de siniestros Graves+Mortales por combinación

In [34]:
# calcular la tasa de siniestros graves o mortales por combinación
part_total = df_hechos.groupby('participantes_siniestro').size().rename('total')
part_grave = (df_hechos[df_hechos['gravedad_siniestro'].isin(['GRAVE','MORTAL'])]
              .groupby('participantes_siniestro').size().rename('grave_mortal'))

tasa_riesgo = pd.concat([part_total, part_grave], axis=1).fillna(0)
tasa_riesgo['tasa_%'] = (tasa_riesgo['grave_mortal'] / tasa_riesgo['total'] * 100).round(2)
tasa_riesgo = tasa_riesgo[tasa_riesgo['total'] >= 50].sort_values('tasa_%', ascending=False).head(15)

fig = px.bar(
    tasa_riesgo.reset_index(), x='participantes_siniestro', y='tasa_%',
    color='tasa_%', color_continuous_scale='Reds',
    labels={'participantes_siniestro': 'Participantes', 'tasa_%': '% Grave o Mortal'},
    title='Tasa de Siniestros Graves/Mortales por Tipo de Participante (mín. 50 hechos)'
)
fig.update_xaxes(tickangle=40)
fig.update_layout(height=460, coloraxis_showscale=False)
fig.show()

### 4️⃣ Pregunta 04 - Evolución temporal 2019–2024

- ¿Cómo varió la cantidad de siniestros a lo largo del tiempo? ¿Se observa el efecto del COVID-19?
- *Hipótesis:* El año 2020 muestra una caída pronunciada en siniestros leves con recuperación progresiva hasta 2024.

##### 4.A - Evolución anual de siniestros por gravedad

In [35]:
anual = (df_hechos
    .groupby(['anio_siniestro', 'gravedad_siniestro'])
    .size()
    .reset_index(name='cantidad')
)

fig = px.line(
    anual, x='anio_siniestro', y='cantidad', color='gravedad_siniestro',
    markers=True,
    color_discrete_map={'LEVE': '#2196F3', 'GRAVE': '#FF9800', 'MORTAL': '#F44336', 'SD': '#9E9E9E'},
    labels={'anio_siniestro': 'Año', 'cantidad': 'Siniestros', 'gravedad_siniestro': 'Gravedad'},
    title='Evolución Anual de Siniestros 2019-2024'
)
fig.add_vline(
    x=2020, line_dash='dash', line_color='gray',
    annotation_text='COVID-19', annotation_position='top right'
)
fig.update_layout(height=440, xaxis=dict(tickmode='linear', dtick=1))
fig.show()

##### 4.B - Distribución mensual promedio (2019-2024)

In [36]:
MESES_ES = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
            7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}

mensual = (df_hechos
    .groupby(['anio_siniestro', 'mes_siniestro'])
    .size()
    .groupby('mes_siniestro')
    .mean()
    .round(1)
    .reset_index()
)
mensual.columns = ['mes', 'promedio']
mensual['mes_label'] = mensual['mes'].map(MESES_ES)

fig = px.bar(
    mensual, x='mes_label', y='promedio',
    color='promedio', color_continuous_scale='Blues',
    labels={'mes_label': 'Mes', 'promedio': 'Promedio Siniestros por Año'},
    title='Promedio Mensual de Siniestros (2019-2024)'
)
fig.update_layout(height=400, coloraxis_showscale=False)
fig.show()

### 5️⃣ Pregunta 05 - Perfil demográfico de las víctimas

- ¿Qué grupos etarios y qué género presentan mayor exposición al riesgo?
- *Hipótesis:* Los hombres jóvenes (18-30 años) concentran la mayor proporción de víctimas mortales.

##### 5.A - Distribución de edad de víctimas por gravedad

In [37]:
df_edad = df_victimas[
    df_victimas['edad_victima'].notna() &
    df_victimas['gravedad'].isin(['LEVE', 'GRAVE', 'MORTAL'])
].copy()

fig = px.violin(
    df_edad, x='gravedad', y='edad_victima',
    color='gravedad', box=True, points='outliers',
    color_discrete_map={'LEVE': '#2196F3', 'GRAVE': '#FF9800', 'MORTAL': '#F44336'},
    category_orders={'gravedad': ['LEVE', 'GRAVE', 'MORTAL']},
    labels={'gravedad': 'Gravedad', 'edad_victima': 'Edad (años)'},
    title='Distribución de Edad de Víctimas por Nivel de Gravedad'
)
fig.update_layout(height=480, showlegend=False)
fig.show()

##### 5.B - Víctimas por sexo y grupo etario según gravedad

In [38]:
# 5.B - Victimas por sexo y grupo etario segun gravedad
# Version robusta para valores reales del dataset (sexo en M/F/SD)
aux = df_victimas.copy()

# Resolver nombres de columnas posibles
sexo_col = 'sexo_victima' if 'sexo_victima' in aux.columns else None
gravedad_col = 'gravedad' if 'gravedad' in aux.columns else ('GRAVEdad_victima' if 'GRAVEdad_victima' in aux.columns else None)
edad_col = 'edad_victima' if 'edad_victima' in aux.columns else None
edad_grupo_col = 'edad_grupo' if 'edad_grupo' in aux.columns else None

if sexo_col is None or gravedad_col is None or edad_col is None:
    raise KeyError(f"Columnas requeridas no encontradas. Disponibles: {aux.columns.tolist()}")

# Si no existe edad_grupo, calcularla localmente para este grafico
if edad_grupo_col is None:
    aux[edad_col] = pd.to_numeric(aux[edad_col], errors='coerce')
    bins = [0, 17, 30, 60, 150]
    labels = ['Menor (<18)', 'Joven (18-30)', 'Adulto (30-60)', 'Mayor (>60)']
    aux['edad_grupo_tmp'] = pd.cut(aux[edad_col], bins=bins, labels=labels, right=True)
    edad_grupo_col = 'edad_grupo_tmp'

# Normalizar valores de sexo (M/F -> MASCULINO/FEMENINO)
aux[sexo_col] = aux[sexo_col].astype(str).str.upper().str.strip()
aux[sexo_col] = aux[sexo_col].replace({
    'M': 'MASCULINO',
    'F': 'FEMENINO',
    'MASC': 'MASCULINO',
    'FEM': 'FEMENINO'
})

# Normalizar gravedad
aux[gravedad_col] = aux[gravedad_col].astype(str).str.upper().str.strip()

sexo_edad = (
    aux[
        aux[sexo_col].isin(['MASCULINO', 'FEMENINO']) &
        aux[edad_grupo_col].notna() &
        aux[gravedad_col].isin(['LEVE', 'GRAVE', 'MORTAL'])
    ]
    .groupby([sexo_col, edad_grupo_col, gravedad_col])
    .size()
    .reset_index(name='cantidad')
    .rename(columns={sexo_col: 'sexo_victima', edad_grupo_col: 'edad_grupo', gravedad_col: 'gravedad'})
)

fig = px.bar(
    sexo_edad,
    x='edad_grupo',
    y='cantidad',
    color='gravedad',
    facet_col='sexo_victima',
    barmode='stack',
    color_discrete_map={'LEVE': '#2196F3', 'GRAVE': '#FF9800', 'MORTAL': '#F44336'},
    category_orders={
        'gravedad': ['LEVE', 'GRAVE', 'MORTAL'],
        'edad_grupo': ['Menor (<18)', 'Joven (18-30)', 'Adulto (30-60)', 'Mayor (>60)'],
        'sexo_victima': ['MASCULINO', 'FEMENINO']
    },
    labels={'edad_grupo': 'Grupo Etario', 'cantidad': 'Victimas', 'gravedad': 'Gravedad', 'sexo_victima': 'Sexo'},
    title='Victimas por Sexo, Grupo Etario y Gravedad'
)
fig.update_layout(height=450)
fig.show()

### 6️⃣ Pregunta 06 - Tipo de vía y gravedad

- ¿Los siniestros en autopistas y avenidas son más graves que en calles secundarias?
- *Hipótesis:* Los siniestros en autopistas presentan mayor proporción de víctimas mortales por efecto de la velocidad.

##### 6.A - Siniestros por tipo de vía según gravedad

In [39]:
via_grav = (df_hechos[df_hechos['tipo_de_via_siniestro'] != 'SD']
    .groupby(['tipo_de_via_siniestro', 'gravedad_siniestro'])
    .size()
    .reset_index(name='cantidad')
)

fig = px.bar(
    via_grav, x='tipo_de_via_siniestro', y='cantidad', color='gravedad_siniestro',
    barmode='stack',
    color_discrete_map={'LEVE': '#2196F3', 'GRAVE': '#FF9800', 'MORTAL': '#F44336', 'SD': '#9E9E9E'},
    category_orders={'gravedad_siniestro': ['LEVE', 'GRAVE', 'MORTAL', 'SD']},
    labels={'tipo_de_via_siniestro': 'Tipo de Vía', 'cantidad': 'Siniestros', 'gravedad_siniestro': 'Gravedad'},
    title='Siniestros por Tipo de Vía y Gravedad'
)
fig.update_layout(height=440)
fig.show()

##### 6.B - Tasa de mortalidad por tipo de vía

In [40]:
via_total   = df_hechos[df_hechos['tipo_de_via_siniestro'] != 'SD'].groupby('tipo_de_via_siniestro').size()
via_mortal  = (df_hechos[
    (df_hechos['tipo_de_via_siniestro'] != 'SD') &
    (df_hechos['gravedad_siniestro'] == 'MORTAL')
].groupby('tipo_de_via_siniestro').size())

tasa_via = pd.DataFrame({'total': via_total, 'mortales': via_mortal}).fillna(0)
tasa_via['tasa_mortal_%'] = (tasa_via['mortales'] / tasa_via['total'] * 100).round(2)
tasa_via = tasa_via.sort_values('tasa_mortal_%', ascending=False).reset_index()

fig = px.bar(
    tasa_via, x='tipo_de_via_siniestro', y='tasa_mortal_%',
    color='tasa_mortal_%', color_continuous_scale='Reds',
    labels={'tipo_de_via_siniestro': 'Tipo de Vía', 'tasa_mortal_%': '% Siniestros Mortales'},
    title='Tasa de Mortalidad (%) por Tipo de Vía'
)
fig.update_layout(height=400, coloraxis_showscale=False)
fig.show()

## 🤖 **Modelo de Machine Learning**

Se entrena un clasificador para predecir la **gravedad** (LEVE / GRAVE / MORTAL) de un siniestro en función de sus características contextuales.

**Variables de entrada:** hora del día, mes, día de la semana, fin de semana, número de comuna.  
**Variable objetivo:** `gravedad_siniestro` (3 clases).  
**Algoritmo:** Random Forest.  
**Tracking:** MLFlow (1 línea de log).

In [41]:
# instalación de librerías adicionales
# depende de cada entorno en particular (Google Colab o VSC en local)
# descomentar según haga falta

# !pip install mlflow -q

In [42]:
# import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder

In [43]:
# preparación de features
FEATURES = ['rango_horario', 'mes_siniestro', 'dia_semana', 'es_fin_de_semana', 'n_comuna']
TARGET   = 'gravedad_siniestro'

df_ml = (df_hechos[FEATURES + [TARGET]]
  .dropna()
  .query(f"{TARGET} in ['LEVE', 'GRAVE', 'MORTAL']")
  .copy()
)

# encoding
le_target = LabelEncoder()
df_ml['target']           = le_target.fit_transform(df_ml[TARGET])
df_ml['es_fin_de_semana'] = df_ml['es_fin_de_semana'].astype(int)
df_ml['n_comuna']         = df_ml['n_comuna'].astype(int)

X = df_ml[FEATURES].astype(float)
y = df_ml['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234, stratify=y)

# entrenamiento con tracking MLFlow
# mlflow.set_experiment("siniestros_viales_gravedad")

# with mlflow.start_run(run_name="RandomForest_v1"):
model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=1234, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

f1 = f1_score(y_test, y_pred, average='weighted')

# mlflow.log_param("n_estimators", 100)
# mlflow.log_param("max_depth", 8)
# mlflow.log_metric("f1_weighted", f1)

In [44]:
print("Reporte de Clasificación")
print(classification_report(y_test, y_pred, target_names=le_target.classes_))
print(f"F1 (weighted): {f1:.4f}")

Reporte de Clasificación
              precision    recall  f1-score   support

       GRAVE       0.00      0.00      0.00       349
        LEVE       0.95      1.00      0.98      9734
      MORTAL       0.00      0.00      0.00       119

    accuracy                           0.95     10202
   macro avg       0.32      0.33      0.33     10202
weighted avg       0.91      0.95      0.93     10202

F1 (weighted): 0.9317


👉🏻 Para ver la UI de MLFlow ejecutar en terminal: `mlflow ui`

In [45]:
# importancia de variables (Random Forest)
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig = px.bar(
    importances.reset_index(),
    x=importances.values,
    y=importances.index,
    orientation='h',
    color=importances.values,
    color_continuous_scale='Blues',
    labels={'x': 'Importancia', 'y': 'Variable'},
    title='Importancia de Variables — Random Forest'
)
fig.update_layout(height=350, coloraxis_showscale=False)
fig.show()

## 🌐 **Demo Interactiva (Gradio + Ngrok)**

Interfaz web para predicción de gravedad en tiempo real.
  
- **Local:** se abre automáticamente en el navegador (puerto 7860)  
- **Ngrok:** genera una URL pública para compartir desde Colab o cualquier entorno

In [46]:
# instalación de librerías adicionales
# depende de cada entorno en particular (Google Colab o VSC en local)
# descomentar según haga falta

# !pip install gradio -q
# !pip install pyngrok -q

In [47]:
import gradio as gr
from pyngrok import ngrok

In [50]:
# predice el nivel de gravedad de un siniestro a partir de sus características
def predecir_gravedad(hora, mes, dia_semana, es_fin_de_semana, comuna):
	features 		= [[ int(hora), int(mes), int(dia_semana), int(es_fin_de_semana), int(comuna) ]]
	pred_idx		= model.predict(features)[0]
	pred_label	= le_target.inverse_transform([pred_idx])[0]
	proba				= model.predict_proba(features)[0]
	proba_dict	= {le_target.classes_[i]: round(float(p), 3) for i, p in enumerate(proba)}
	return pred_label, proba_dict

In [ ]:
COMUNAS = [str(i) for i in range(1, 16)]

with gr.Blocks(title="Predictor Siniestros Viales - CABA") as demo:
	gr.Markdown("## 🚗 Predictor de Gravedad - Siniestros Viales CABA")
	gr.Markdown("Ingresá las condiciones del siniestro para estimar su nivel de gravedad.")

	with gr.Row():
		hora       = gr.Slider(0, 23, step=1, value=12, label="Hora del día (0-23)")
		mes        = gr.Slider(1, 12,  step=1, value=6,  label="Mes (1=Ene ... 12=Dic)")
		dia_semana = gr.Slider(0, 6,   step=1, value=2,  label="Día semana (0=Lun, 6=Dom)")

	with gr.Row():
		fin_semana = gr.Checkbox(label="¿Es fin de semana?", value=False)
		comuna     = gr.Dropdown(choices=COMUNAS, value="4", label="Comuna (1-15)")

	btn = gr.Button("🔍 Predecir gravedad", variant="primary")

	with gr.Row():
		pred_label = gr.Textbox(label="Nivel de gravedad predicho")
		pred_proba = gr.JSON(label="Probabilidades por clase")

	btn.click(
		fn = predecir_gravedad,
		inputs = [hora, mes, dia_semana, fin_semana, comuna],
		outputs = [pred_label, pred_proba]
	)

# ejecutar la demo
try:
	# ngrok.set_auth_token("3C124nYd9yKnBjAbyuGDFGQGOer_6hzAHnGorTkP68Yg2JjWL")
	public_url = ngrok.connect(7860) 						
	print(f"URL pública (Ngrok): { public_url }")							# URL pública de Ngrok

except ImportError:
	print(f"URL pública (Ngrok): No Disponible (ejecutando solamente en modo local)")

finally:
	demo.launch(server_port=7860, share=False, quiet=False)		# sin Ngrok, abrir en localhost

URL pública (Ngrok): NgrokTunnel: "https://interpressure-brennan-drivelingly.ngrok-free.dev" -> "http://localhost:7860"
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 📝 **Conclusiones**

Algunos de los hallazgos respecto a las hipótesis planteadas:

1. **Franjas Horarias:**
	- Hipótesis: La madrugada (0-5h) concentra la mayor proporción de siniestros graves y mortales, especialmente los fines de semana.
	- Hallazgo: Madrugada = 11% grave/mortal (vs 4,1% tarde). Similar fin de semana y días hábiles.

2. **Comunas:**
	- Hipótesis: Las comunas con mayor flujo (1, 4, 9) lideran en cantidad absoluta, pero la tasa de mortalidad relativa puede ser mayor en comunas periféricas.
	- Hallazgo: Comunas	C1 lidera en volumen (5.841), pero C8 tiene mayor tasa mortalidad (1,98%)

3. **Participantes:**
	- Hipótesis: Las combinaciones MOTO-AUTO y PEATON-AUTO representan la mayor carga de siniestros graves/mortales.
	- Hallazgo: MOTO-AUTO domina en volumen (11.921), MOTO-OBJETO FIJO lidera en tasa (25,9%)

4. **Evolución Temporal:**
	- Hipótesis: Se observa una caída marcada en 2020 (COVID-19) con un incremento progresiva hasta 2024.
	- Hallazgo: Caída del 40% en 2020 (COVID), superación del nivel pre-pandemia en 2024

5. **Perfil Demográfico:**
	- Hipótesis: Los hombres jóvenes (18-30 años) son el grupo más afectado en siniestros mortales.
	- Hallazgo: Hombres = 74,7% de muertes. Adultos 30-60 superan a jóvenes 18-30

6. **Tipo de Vía:**
	- Hipótesis: Las autopistas muestran mayor tasa de mortalidad relativa que las calles, confirmando el efecto de la velocidad.
	- Hallazgo: Autopistas = 8,62% mortalidad (~10x más que calles)

7. **Adicional**
	- Del modelo entrenado: F1=0.93 pero recall 0% en GRAVE/MORTAL debido al desbalance de clases del dataset
